In [1]:
%pip install opencv-python numpy pillow pytesseract easyocr

  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached torchvision-0.29.1-cp313-cp313-win_amd64.whl.metadata (5.7 kB)
  Using cached opencv_python_headless-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached ninja-1.13.2-py3-none-win_amd64.whl.metadata (5.4 kB)
  Using cached torch-2.14.1-cp313-cp313-win_amd64.whl.metadata (38 kB)
Using cached easyocr-1.7.2-py3-none-any.whl (2.9 MB)
Using cached torchvision-0.29.1-cp313-cp313-win_amd64.whl (1.4 MB)
Using cached torch-2.14.1-cp313-cp313-win_amd64.whl (124.1 MB)
Using cached ninja-1.13.2-py3-none-win_amd64.whl (311 kB)
Using cached opencv_python_headless-5.0.0.93-cp37-abi3-win_amd64.whl (43.8 MB)

   ---------------------------------------- 0/5 [opencv-python-headless]

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\koda\\anaconda3\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [3]:
%%writefile module4_opencv.py
import cv2
import numpy as np


def load_image(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return img


def upscale_if_small(img, min_width=1000):
    h, w = img.shape[:2]
    if w < min_width:
        s = min_width / w
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    return img


def deskew(gray):
    inv = cv2.bitwise_not(gray)
    coords = np.column_stack(np.where(inv > 0))
    if len(coords) < 50:
        return gray
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.5:
        return gray
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def preprocess(img, denoise=True, fix_skew=True, threshold=True):
    img = upscale_if_small(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if denoise:
        gray = cv2.fastNlMeansDenoising(gray, h=10)
    if fix_skew:
        gray = deskew(gray)
    if threshold:
        gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    return gray

Writing module4_opencv.py


In [4]:
from module4_opencv import load_image, preprocess

In [7]:
import os

home = os.path.expanduser("~")
found = []
for base in ["Downloads", "Desktop", "Documents"]:
    for root, dirs, files in os.walk(os.path.join(home, base)):
        if "module4_opencv.py" in files:
            found.append(os.path.dirname(root))   # the folder that contains "modules"
            break
        if root.count(os.sep) - home.count(os.sep) > 6:   # don't search too deep
            dirs.clear()

print(found)

['C:\\Users\\koda']


In [10]:
from module4_opencv import load_image, preprocess

In [14]:
%pip install easyocr

  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached torchvision-0.29.1-cp313-cp313-win_amd64.whl.metadata (5.7 kB)
  Using cached opencv_python_headless-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached ninja-1.13.2-py3-none-win_amd64.whl.metadata (5.4 kB)
  Using cached torch-2.14.1-cp313-cp313-win_amd64.whl.metadata (38 kB)
Using cached easyocr-1.7.2-py3-none-any.whl (2.9 MB)
Using cached torchvision-0.29.1-cp313-cp313-win_amd64.whl (1.4 MB)
Using cached torch-2.14.1-cp313-cp313-win_amd64.whl (124.1 MB)
Using cached ninja-1.13.2-py3-none-win_amd64.whl (311 kB)
Using cached opencv_python_headless-5.0.0.93-cp37-abi3-win_amd64.whl (43.8 MB)

   ---------------------------------------- 0/5 [opencv-python-headless]

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\koda\\anaconda3\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [16]:

import cv2

def preprocess(img):
    h, w = img.shape[:2]

    # Upscale the image
    scale = max(2.0, 1800 / w)
    img = cv2.resize(
        img, None,
        fx=scale, fy=scale,
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Improve local contrast
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    gray = clahe.apply(gray)

    # Reduce noise while preserving edges
    gray = cv2.bilateralFilter(gray, 5, 50, 50)

    # Return grayscale first; thresholding can remove fine text
    return gray


In [17]:

cleaned = preprocess(load_image(str(image_path)))

for psm in [3, 6, 11, 12]:
    config = f"--oem 3 --psm {psm}"

    text = pytesseract.image_to_string(
        cleaned,
        lang="eng",
        config=config
    ).strip()

    print(f"\n{'=' * 20} PSM {psm} {'=' * 20}")
    print(text if text else "No text detected")



==================== PSM 3 ====================
No text detected

==================== PSM 6 ====================
peal

low

‘
=

is

> ~Ay : 3 < . ~y
: a a . Va gl a i :
—£" s z a \ - n st ale

==================== PSM 11 ====================
=

a)

s¥

NS

a

mA

—_—-

ie

i

Wwe |

WH i-

-_

—

h

~Te

ih

A

}*

a*%

are

AL ae Pt

Ne |

==================== PSM 12 ====================
=

a)

s¥

NS

a

mA

Ss

—_—-

’

Wwe |

WH i-

ih

-

h

:

és ee

:

Ne |


In [18]:

ground_truth = """
ACME SOLUTIONS
Innovate Build Grow
Invoice No: INV-2024-001
Date: 15 April 2024
Due Date: 30 April 2024
Description
Web Development Services
Technical Support (Monthly)
Consulting Services
Subtotal 2600.00
Tax 260.00
Total (USD) 2860.00
"""


In [19]:

from jiwer import cer, wer

predicted_text = text  # Output from your OCR function

character_error = cer(ground_truth, predicted_text)
word_error = wer(ground_truth, predicted_text)

print(f"CER: {character_error:.2%}")
print(f"WER: {word_error:.2%}")
print(f"Approximate character accuracy: {max(0, 1-character_error):.2%}")
print(f"Approximate word accuracy: {max(0, 1-word_error):.2%}")


CER: 88.56%
WER: 100.00%
Approximate character accuracy: 11.44%
Approximate word accuracy: 0.00%


In [21]:

from pathlib import Path
from jiwer import cer, wer

# Reuse the image path you selected earlier
text, conf, engine = extract_text(str(image_path))

# Use your existing ground_truth variable
character_error = cer(ground_truth, text)
word_error = wer(ground_truth, text)

report = f"""OCR Evaluation Report

Engine: {engine}
Confidence: {conf:.1f}%
CER: {character_error:.2%}
WER: {word_error:.2%}

Extracted text:
{text}
"""

Path("ocr_evaluation_report.txt").write_text(
    report, encoding="utf-8"
)

print(report)
print("\nSaved report to:", Path("ocr_evaluation_report.txt").resolve())


EasyOCR fallback failed: No module named 'easyocr'
OCR Evaluation Report

Engine: tesseract
Confidence: 38.8%
CER: 86.86%
WER: 122.73%

Extracted text:
peal

low

‘
=

is

> ~Ay : 3 < . ~y
: a a . Va gl a i :
—£" s z a \ - n st ale


Saved report to: C:\Users\koda\Downloads\ocr_evaluation_report.txt


In [15]:

import cv2
import numpy as np
import pytesseract
from pathlib import Path

# Configure Tesseract on Windows if it is installed here.
TESSERACT_PATH = Path(
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

if TESSERACT_PATH.is_file():
    pytesseract.pytesseract.tesseract_cmd = str(TESSERACT_PATH)

_easy_reader = None


def load_image(path):
    """Load an image and validate its path."""
    path = Path(path).expanduser()

    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    image = cv2.imread(str(path))

    if image is None:
        raise ValueError(f"OpenCV cannot read: {path}")

    return image


def preprocess(img):
    """Prepare the image for OCR."""
    h, w = img.shape[:2]

    if w < 1000:
        scale = 1000 / w
        img = cv2.resize(
            img, None, fx=scale, fy=scale,
            interpolation=cv2.INTER_CUBIC
        )

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.fastNlMeansDenoising(gray, h=10)

    return cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]


def tesseract_ocr(img):
    """Return OCR text and average word confidence."""
    config = "--oem 3 --psm 6"

    text = pytesseract.image_to_string(
        img, lang="eng", config=config
    ).strip()

    data = pytesseract.image_to_data(
        img,
        lang="eng",
        config=config,
        output_type=pytesseract.Output.DICT
    )

    confs = []

    for word, conf in zip(data["text"], data["conf"]):
        try:
            value = float(conf)
            if word.strip() and value >= 0:
                confs.append(value)
        except (ValueError, TypeError):
            continue

    confidence = sum(confs) / len(confs) if confs else 0.0
    return text, confidence


def easyocr_ocr(path):
    """Fallback OCR using EasyOCR."""
    global _easy_reader

    import easyocr

    if _easy_reader is None:
        _easy_reader = easyocr.Reader(["en"], gpu=False)

    results = _easy_reader.readtext(str(path))
    text = "\n".join(result[1] for result in results)

    confidence = (
        100 * sum(result[2] for result in results) / len(results)
        if results else 0.0
    )

    return text, confidence


def extract_text(path, min_conf=60):
    """Return (text, confidence, engine)."""
    image = load_image(path)
    cleaned = preprocess(image)

    text, conf = tesseract_ocr(cleaned)

    if text and conf >= min_conf:
        return text, conf, "tesseract"

    try:
        text2, conf2 = easyocr_ocr(path)

        if text2.strip() and (not text.strip() or conf2 > conf):
            return text2, conf2, "easyocr"

    except Exception as error:
        print("EasyOCR fallback failed:", error)

    return text, conf, "tesseract"


# Run this cell directly in Jupyter Notebook.
# Enter the real path to your image when prompted.
image_path = input(
    "Enter your image's full path: "
).strip().strip('"')

try:
    text, conf, engine = extract_text(image_path)

    print(f"\nEngine: {engine}")
    print(f"Confidence: {conf:.1f}%")
    print("\nExtracted text:\n")
    print(text if text else "No text detected.")

    Path("extracted_text.txt").write_text(
        text, encoding="utf-8"
    )
    print("\nSaved: extracted_text.txt")

except Exception as error:
    print(f"\nERROR: {error}")


EasyOCR fallback failed: No module named 'easyocr'

Engine: tesseract
Confidence: 57.0%

Extracted text:

oan =

Saved: extracted_text.txt


In [23]:

# Use your existing image path
print("Image path:", image_path)

# Replace this sample with the exact text in your image
ground_truth = """
ACME SOLUTIONS
Invoice No: INV-2024-001
Description
Web Development Services
Technical Support
Consulting Services
Subtotal
Tax
Total
"""


Image path: C:\Users\koda\OneDrive\Pictures\WhatsApp Image 2024-09-03 at 8.30.24 PM.jpeg


In [24]:

import cv2
import pandas as pd
import pytesseract
from jiwer import cer, wer

# Load the image once
original = load_image(str(image_path))

def tune_preprocess(image, width, denoise, threshold):
    h, w = image.shape[:2]

    if w < width:
        scale = width / w
        image = cv2.resize(
            image, None, fx=scale, fy=scale,
            interpolation=cv2.INTER_CUBIC
        )

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if denoise:
        gray = cv2.fastNlMeansDenoising(gray, h=10)

    if threshold == "otsu":
        gray = cv2.threshold(
            gray, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )[1]
    elif threshold == "adaptive":
        gray = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 31, 11
        )

    return gray

results = []

for width in [1000, 1400, 1800]:
    for denoise in [False, True]:
        for threshold in ["none", "otsu"]:
            for psm in [3, 6, 11]:
                processed = tune_preprocess(
                    original, width, denoise, threshold
                )

                config = f"--oem 3 --psm {psm}"

                text = pytesseract.image_to_string(
                    processed, lang="eng", config=config
                ).strip()

                results.append({
                    "width": width,
                    "denoise": denoise,
                    "threshold": threshold,
                    "psm": psm,
                    "CER": cer(ground_truth, text),
                    "WER": wer(ground_truth, text),
                    "text": text
                })

results_df = pd.DataFrame(results).sort_values(
    ["CER", "WER"]
).reset_index(drop=True)

print("Top 10 OCR configurations:")
display(results_df.drop(columns="text").head(10))

best = results_df.iloc[0]

print("\nBest configuration:")
print(best[["width", "denoise", "threshold", "psm", "CER", "WER"]])

print("\nBest extracted text:")
print(best["text"])

with open("best_ocr_output.txt", "w", encoding="utf-8") as f:
    f.write(best["text"])

results_df.drop(columns="text").to_csv(
    "ocr_tuning_results.csv", index=False
)

print("\nSaved best_ocr_output.txt")
print("Saved ocr_tuning_results.csv")


Top 10 OCR configurations:


,width,denoise,threshold,psm,CER,WER
0,1800,False,otsu,11,0.909774,1.000
1,1800,False,none,6,0.917293,1.125
2,1800,False,otsu,6,0.917293,1.125
3,1800,False,none,11,0.939850,1.000
4,1800,True,otsu,6,0.939850,1.000
5,1000,False,none,6,0.947368,1.000
6,1400,False,none,6,0.947368,1.000
7,1000,False,otsu,6,0.962406,1.000
8,1400,False,otsu,6,0.962406,1.000
9,1800,True,none,6,0.962406,1.000



Best configuration:
width            1800
denoise         False
threshold        otsu
psm                11
CER          0.909774
WER               1.0
Name: 0, dtype: object

Best extracted text:
X

W .

vi

a

es

Saved best_ocr_output.txt
Saved ocr_tuning_results.csv


In [26]:
import json

best = results_df.iloc[0]

best_config = {
    "width": int(best["width"]),
    "denoise": bool(best["denoise"]),
    "threshold": str(best["threshold"]),
    "psm": int(best["psm"]),
    "CER": float(best["CER"]),
    "WER": float(best["WER"])
}

with open("best_ocr_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, indent=4)

print(json.dumps(best_config, indent=4))
print("\nSaved best_ocr_config.json")

{
    "width": 1800,
    "denoise": false,
    "threshold": "otsu",
    "psm": 11,
    "CER": 0.9097744360902256,
    "WER": 1.0
}

Saved best_ocr_config.json
